<a href="https://colab.research.google.com/github/diksha11-12/ai-quiz-generator/blob/main/ai-quiz-generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U gradio
!pip install -q --pre fireworks-ai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 75.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.3/468.3 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 72.8 MB/s eta 0:00:00


In [ ]:
import json
import time
import html
import gradio as gr
from fireworks import Fireworks
from google.colab import userdata

api_key = userdata.get("FIREWORK_API_KEY")

client = Fireworks(
    api_key=api_key,
    account_id="fireworks",
) if api_key else None


def get_available_models():
    if not client:
        return [], "⚠️ FIREWORKS_API_KEY not found in Colab Secrets."

    try:
        models = list(
            client.models.list(filter="supports_serverless=true")
        )

        names = sorted(model.name for model in models)

        if not names:
            return [], "⚠️ No Fireworks serverless models are available."

        return names, f"✅ Loaded {len(names)} Fireworks models."

    except Exception as e:
        return [], f"⚠️ Could not load models: {html.escape(str(e))}"


def refresh_models(): # This function is no longer used but kept for completeness if needed elsewhere.
    models, message = get_available_models()
    return gr.update(choices=models, value=models[0] if models else None), message


def clean_json(text):
    text = (text or "").strip()

    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        text = text.rsplit("```", 1)[0].strip()

    return json.loads(text)


def generate_quiz(topic, count, difficulty, model):
    if not (topic := (topic or "").strip()):
        return [], {}, "⚠️ Please enter a quiz topic."

    if not client:
        return [], {}, "⚠️ FIREWORKS_API_KEY not found in Colab Secrets."

    if not model:
        return [], {}, "⚠️ No model selected. Please ensure a Fireworks API key is set and models are available."

    prompt = f"""Generate exactly {int(count)} unique multiple-choice questions about "{topic}" at {difficulty} difficulty.\n\nReturn ONLY valid JSON:\n{{\n  "quiz": [\n    {{\n      "question": "...",\n      "options": {{\n        "A": "...",\n        "B": "...",\n        "C": "...",\n        "D": "..."\n      }},\n      "answer": "A"\n    }}\n  ]\n}}"""

    try:
        for attempt in range(3):
            try:
                response = client.chat.completions.create(
                    model=model,
                    messages=[
                        {
                            "role": "system",
                            "content": "You create accurate educational quizzes. Return JSON only.",
                        },
                        {"role": "user", "content": prompt},
                    ],
                    temperature=0.7,
                )
                break
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(2 ** attempt)

        data = clean_json(response.choices[0].message.content)
        quiz = data["quiz"]
        valid_answers = {"A", "B", "C", "D"}

        if not isinstance(quiz, list) or len(quiz) != int(count):
            raise ValueError("Incorrect number of questions returned.")

        for question in quiz:
            if (
                not isinstance(question.get("question"), str)
                or set(question.get("options", {}).keys()) != valid_answers
                or question.get("answer") not in valid_answers
            ):
                raise ValueError("Invalid quiz format returned.")

        return quiz, {}, "✅ Quiz generated successfully. Answer all questions."

    except json.JSONDecodeError:
        return [], {}, "⚠️ Invalid JSON received. Please generate again."
    except Exception as e:
        return [], {}, f"⚠️ Error: {html.escape(str(e))}"


def save_answer(choice, answers, index):
    answers = dict(answers or {})

    if choice:
        answers[index] = choice.split(".", 1)[0]

    return answers


def submit_quiz(quiz, answers):
    if not quiz:
        return "", "⚠️ Generate a quiz first."

    answers = answers or {}
    missing = [str(i + 1) for i in range(len(quiz)) if i not in answers]

    if missing:
        return "", f"⚠️ Answer every question first. Missing: {', '.join(missing)}"

    score = sum(
        answers[i] == question["answer"]
        for i, question in enumerate(quiz)
    )

    percentage = score / len(quiz) * 100
    passed = percentage >= 60
    status = "PASS 🎉" if passed else "FAIL ❌"
    color = "#16a34a" if passed else "#dc2626"

    review = []

    for i, question in enumerate(quiz):
        selected = answers[i]
        correct = question["answer"]
        icon = "✅" if selected == correct else "❌"

        review.append(f"""
        <div class="review-card">
            <b>Question {i + 1}: {html.escape(question["question"])}</b><br>
            {icon} Your answer:
            <b>{selected}. {html.escape(question["options"][selected])}</b><br>
            Correct answer:
            <b>{correct}. {html.escape(question["options"][correct])}</b>
        </div>
        """)

    result = f"""
    <div class="score-card">
        <h2>Score: {score} / {len(quiz)}</h2>
        <p>Percentage: <b>{percentage:.1f}%</b></p>
        <p style="color:{color}; font-size:18px"><b>{status}</b></p>
    </div>
    <h2 class="review-title">Answer Review</h2>
    {''.join(review)}
    """

    return result, "✅ Quiz submitted successfully."


def reset_quiz():
    return "", 5, "Moderate", [], {}, "Enter a topic and generate a new quiz."

# Get default model without exposing dropdown choices
_all_models, _ = get_available_models()
_default_model = _all_models[0] if _all_models else None
_initial_message = "Ready to generate a quiz. Make sure FIREWORKS_API_KEY is set in Colab Secrets." if not _default_model else "Ready to generate a quiz."
if not _all_models:
    _initial_message = "⚠️ No Fireworks serverless models are available or FIREWORKS_API_KEY not found."

css = """
body {
    background: linear-gradient(135deg, #dbeafe, #f3e8ff) !important;
}

.gradio-container {
    width: 100% !important;
    min-height: 100vh;
    background: linear-gradient(135deg, #dbeafe, #f3e8ff) !important;
}

.quiz-card,
.review-card,
.score-card {
    background: #ffffff !important;
    color: #111827 !important;
    border: 1px solid #c7d2fe;
    border-radius: 16px;
    padding: 18px;
    margin: 14px 0;
    box-shadow: 0 5px 14px rgba(79, 70, 229, 0.12);
}

.quiz-card {
    border-left: 6px solid #6366f1;
}

.review-card {
    border-left: 6px solid #8b5cf6;
}

.score-card {
    border-left: 6px solid #4f46e5;
    background: #eef2ff !important;
}

.quiz-card h3,
.quiz-card p,
.review-card,
.review-card b,
.score-card,
.score-card h2,
.score-card p,
.review-title {
    color: #111827 !important;
}

h1, h2, h3 {
    color: #312e81 !important;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="indigo"),
    css=css,
    title="Fireworks AI Quiz Generator",
) as demo:

    gr.HTML("""
    <div style="text-align:center; color:#312e81; padding:10px">
        <h1>🧠 Fireworks AI Quiz Generator</h1>
        <p>Create and solve AI-generated quizzes.</p>
    </div>
    """)

    quiz_state = gr.State([])
    answers_state = gr.State({})
    # Hidden state to store the default model for quiz generation
    selected_model_hidden_state = gr.State(_default_model)

    with gr.Row():
        topic = gr.Textbox(
            label="Quiz Topic",
            placeholder="Example: Python, Java, Machine Learning",
            scale=3,
        )

        count = gr.Slider(
            minimum=1,
            maximum=10,
            value=5,
            step=1,
            label="Questions",
            scale=1,
        )

    difficulty = gr.Radio(
        choices=["Easy", "Moderate", "Difficult"],
        value="Moderate",
        label="Difficulty",
    )

    with gr.Row():
        generate_btn = gr.Button("🚀 Generate Quiz", variant="primary")
        new_btn = gr.Button("New Quiz")

    message = gr.Markdown(_initial_message)

    @gr.render(inputs=[quiz_state])
    def show_questions(quiz):
        if not quiz:
            return

        gr.Markdown("## 📝 Your Quiz")

        for i, question in enumerate(quiz):
            with gr.Group():
                gr.HTML(f"""
                <div class="quiz-card">
                    <h3>Question {i + 1}</h3>
                    <p>{html.escape(question["question"])}</p>
                </div>
                """)

                radio = gr.Radio(
                    choices=[
                        f"{key}. {question['options'][key]}"
                        for key in "ABCD"
                    ],
                    label="Select your answer",
                )

                radio.change(
                    lambda choice, answers, index=i: save_answer(
                        choice, answers, index
                    ),
                    inputs=[radio, answers_state],
                    outputs=answers_state,
                )

        submit_btn = gr.Button("Submit Quiz", variant="primary")
        result = gr.HTML()

        submit_btn.click(
            submit_quiz,
            inputs=[quiz_state, answers_state],
            outputs=[result, message],
        )

    # Removed refresh_btn.click handler

    generate_btn.click(
        generate_quiz,
        inputs=[topic, count, difficulty, selected_model_hidden_state], # Use the hidden state for the model
        outputs=[quiz_state, answers_state, message],
        show_progress="full",
    )

    new_btn.click(
        reset_quiz,
        outputs=[
            topic,
            count,
            difficulty,
            quiz_state,
            answers_state,
            message,
        ],
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_1590/3975975333.py:227: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1107ca9c84edaa6c77.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
